# Lyrics Training Notebook

This notebook trains a text generation model on lyrics stored in Azure Blob Storage.

## Features:
- Load lyrics from Azure Blob Storage
- Preprocess text data
- Train a simple LSTM-based text generation model
- Export and register the model in Azure ML
- Optional: Deploy to Azure ML endpoint

## Prerequisites:
- Azure ML Workspace
- Azure Blob Storage with lyrics text file
- Required Python packages (installed in setup cell)

## 1. Setup and Configuration

In [ ]:
# Install required packages
!pip install azureml-core azure-storage-blob tensorflow numpy pandas scikit-learn

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from azureml.core import Workspace, Dataset, Experiment, Model, Environment
from azureml.core.model import InferenceConfig
from azureml.core.webservice import AciWebservice
from azure.storage.blob import BlobServiceClient
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print("TensorFlow version:", tf.__version__)
print("Azure ML SDK version:", azureml.core.VERSION)

In [ ]:
# ============================================================================
# AZURE CONFIGURATION - SET THESE VALUES BEFORE RUNNING
# ============================================================================

# Azure ML Workspace Configuration
AZURE_SUBSCRIPTION_ID = os.getenv('AZURE_SUBSCRIPTION_ID', 'YOUR_SUBSCRIPTION_ID')
AZURE_RESOURCE_GROUP = os.getenv('AZURE_RESOURCE_GROUP', 'YOUR_RESOURCE_GROUP')
AZURE_WORKSPACE_NAME = os.getenv('AZURE_WORKSPACE_NAME', 'YOUR_WORKSPACE_NAME')

# Azure Storage Configuration
STORAGE_ACCOUNT_NAME = os.getenv('STORAGE_ACCOUNT_NAME', 'YOUR_STORAGE_ACCOUNT')
STORAGE_ACCOUNT_KEY = os.getenv('STORAGE_ACCOUNT_KEY', 'YOUR_STORAGE_KEY')
AZURE_STORAGE_CONNECTION_STRING = os.getenv('AZURE_STORAGE_CONNECTION_STRING', '')
BLOB_CONTAINER_NAME = os.getenv('BLOB_CONTAINER_NAME', 'lyrics')
BLOB_NAME = os.getenv('BLOB_NAME', 'lyrics.txt')

# Model Configuration
MODEL_NAME = 'lyrics-text-generation-model'
EXPERIMENT_NAME = 'lyrics-training-experiment'

# Training Configuration
EPOCHS = 50
BATCH_SIZE = 64

# Deployment Configuration
DEPLOY_MODEL = False
DEPLOYMENT_SERVICE_NAME = 'lyrics-generation-service'

# ============================================================================
print("✓ Azure Configuration Loaded")
print(f"  Workspace: {AZURE_WORKSPACE_NAME}")
print(f"  Storage Container: {BLOB_CONTAINER_NAME}/{BLOB_NAME}")
print(f"  Deploy Model: {DEPLOY_MODEL}")

## 2. Connect to Azure ML Workspace

In [ ]:
# Connect to Azure ML Workspace using global configuration variables
try:
    ws = Workspace.from_config()
    print(f"Loaded workspace: {ws.name}")
except:
    # Fallback: Create workspace object manually using globals
    ws = Workspace(subscription_id=AZURE_SUBSCRIPTION_ID,
                   resource_group=AZURE_RESOURCE_GROUP,
                   workspace_name=AZURE_WORKSPACE_NAME)
    print(f"Connected to workspace: {ws.name}")

print(f"\nWorkspace details:")
print(f"  Name: {ws.name}")
print(f"  Resource Group: {ws.resource_group}")
print(f"  Location: {ws.location}")

## 3. Load Lyrics from Azure Blob Storage

In [ ]:
# Load lyrics from blob storage using global configuration variables

def load_lyrics_from_blob(connection_string, container_name, blob_name, storage_account_name, storage_account_key):
    """Load lyrics text file from Azure Blob Storage"""
    try:
        if connection_string:
            blob_service_client = BlobServiceClient.from_connection_string(connection_string)
        else:
            # Use account name and key
            account_url = f"https://{storage_account_name}.blob.core.windows.net"
            blob_service_client = BlobServiceClient(account_url=account_url, 
                                                   credential=storage_account_key)
        
        blob_client = blob_service_client.get_blob_client(container=container_name, 
                                                          blob=blob_name)
        
        # Download blob content
        blob_data = blob_client.download_blob()
        text_content = blob_data.readall().decode('utf-8')
        
        print(f"Successfully loaded lyrics from {container_name}/{blob_name}")
        print(f"Text length: {len(text_content)} characters")
        
        return text_content
    except Exception as e:
        print(f"Error loading lyrics: {str(e)}")
        # Fallback to sample data for demonstration
        print("Using sample lyrics data...")
        return """This is a sample lyrics line\nAnother line of lyrics\nMusic and melodies\nSinging together\n""" * 10

lyrics_text = load_lyrics_from_blob(AZURE_STORAGE_CONNECTION_STRING, 
                                     BLOB_CONTAINER_NAME, 
                                     BLOB_NAME,
                                     STORAGE_ACCOUNT_NAME,
                                     STORAGE_ACCOUNT_KEY)
print(f"\nFirst 500 characters:\n{lyrics_text[:500]}")

## 4. Preprocess Lyrics Data

In [ ]:
# Split into lines and clean
lines = [line.strip() for line in lyrics_text.split('\n') if line.strip()]
print(f"Total lines: {len(lines)}")
print(f"Sample lines:")
for i, line in enumerate(lines[:5]):
    print(f"  {i+1}. {line}")

In [ ]:
# Tokenize the text
tokenizer = Tokenizer(char_level=False, lower=True)
tokenizer.fit_on_texts(lines)
total_words = len(tokenizer.word_index) + 1

print(f"Total unique words: {total_words}")
print(f"Vocabulary size: {len(tokenizer.word_index)}")

In [ ]:
# Create input sequences
input_sequences = []
for line in lines:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total sequences: {len(input_sequences)}")

# Pad sequences
max_sequence_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, 
                                         maxlen=max_sequence_len, 
                                         padding='pre'))

print(f"Max sequence length: {max_sequence_len}")
print(f"Input sequences shape: {input_sequences.shape}")

In [ ]:
# Create predictors and labels
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encode the labels
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

## 5. Build and Train the Model

In [ ]:
# Create an experiment using global configuration
experiment = Experiment(workspace=ws, name=EXPERIMENT_NAME)
print(f"Experiment: {experiment.name}")

In [ ]:
# Start a run
run = experiment.start_logging()

# Log parameters
run.log('vocab_size', total_words)
run.log('max_sequence_length', max_sequence_len)
run.log('total_sequences', len(input_sequences))

print("Started experiment run")

In [ ]:
# Build the model
model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', 
              optimizer='adam', 
              metrics=['accuracy'])

print(model.summary())

In [ ]:
# Train the model using global configuration
# Callbacks
checkpoint = ModelCheckpoint('best_model.h5', 
                            monitor='loss', 
                            save_best_only=True, 
                            mode='min')
early_stop = EarlyStopping(monitor='loss', 
                          patience=5, 
                          restore_best_weights=True)

history = model.fit(X, y, 
                   epochs=EPOCHS, 
                   batch_size=BATCH_SIZE,
                   callbacks=[checkpoint, early_stop],
                   verbose=1)

# Log metrics
run.log('final_loss', history.history['loss'][-1])
run.log('final_accuracy', history.history['accuracy'][-1])
run.log('epochs_trained', len(history.history['loss']))

print("Training completed!")

## 6. Test the Model

In [ ]:
# Generate text function
def generate_text(seed_text, next_words, model, max_sequence_len, tokenizer):
    """Generate text based on seed text"""
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], 
                                   maxlen=max_sequence_len-1, 
                                   padding='pre')
        predicted = model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted, axis=-1)[0]
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                output_word = word
                break
        
        seed_text += " " + output_word
    
    return seed_text

# Test generation
print("Testing text generation:")
print("\nSeed: 'love'")
print(generate_text("love", 10, model, max_sequence_len, tokenizer))

print("\nSeed: 'music'")
print(generate_text("music", 10, model, max_sequence_len, tokenizer))

## 7. Export and Register the Model

In [ ]:
# Create model directory
model_dir = 'outputs'
os.makedirs(model_dir, exist_ok=True)

# Save the model
model_path = os.path.join(model_dir, 'lyrics_model.h5')
model.save(model_path)
print(f"Model saved to {model_path}")

# Save tokenizer
tokenizer_config = {
    'word_index': tokenizer.word_index,
    'max_sequence_len': int(max_sequence_len),
    'total_words': int(total_words)
}
tokenizer_path = os.path.join(model_dir, 'tokenizer_config.json')
with open(tokenizer_path, 'w') as f:
    json.dump(tokenizer_config, f)
print(f"Tokenizer config saved to {tokenizer_path}")

In [ ]:
# Register the model in Azure ML using global configuration
registered_model = Model.register(
    workspace=ws,
    model_path=model_dir,
    model_name=MODEL_NAME,
    tags={
        'type': 'text-generation',
        'domain': 'lyrics',
        'framework': 'tensorflow'
    },
    description='LSTM-based text generation model trained on lyrics'
)

print(f"Model registered: {registered_model.name}")
print(f"Version: {registered_model.version}")
print(f"ID: {registered_model.id}")

# Log model in the run
run.upload_folder('outputs', model_dir)
run.register_model(
    model_name=MODEL_NAME,
    model_path='outputs'
)

# Complete the run
run.complete()
print("Run completed")

## 8. (Optional) Deploy Model to Azure ML Endpoint

This section allows you to deploy the trained model to an Azure ML endpoint for real-time inference.

In [ ]:
# Set this to True to deploy the model
DEPLOY_MODEL = False

if DEPLOY_MODEL:
    print("Preparing model deployment...")
else:
    print("Model deployment skipped. Set DEPLOY_MODEL=True to deploy.")

In [ ]:
# Create scoring script
if DEPLOY_MODEL:
    scoring_script = '''
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import os

def init():
    global model, tokenizer_config
    
    model_path = os.path.join(os.getenv('AZUREML_MODEL_DIR'), 'lyrics_model.h5')
    tokenizer_path = os.path.join(os.getenv('AZUREML_MODEL_DIR'), 'tokenizer_config.json')
    
    model = load_model(model_path)
    
    with open(tokenizer_path, 'r') as f:
        tokenizer_config = json.load(f)
    
    print("Model and tokenizer loaded successfully")

def run(raw_data):
    try:
        data = json.loads(raw_data)
        seed_text = data.get('seed_text', 'love')
        next_words = data.get('next_words', 10)
        
        # Generate text
        word_index = tokenizer_config['word_index']
        max_sequence_len = tokenizer_config['max_sequence_len']
        
        for _ in range(next_words):
            # Convert text to sequence
            token_list = []
            for word in seed_text.lower().split():
                if word in word_index:
                    token_list.append(word_index[word])
            
            # Pad sequence
            token_list = pad_sequences([token_list], 
                                      maxlen=max_sequence_len-1, 
                                      padding='pre')
            
            # Predict
            predicted = model.predict(token_list, verbose=0)
            predicted_word_index = np.argmax(predicted, axis=-1)[0]
            
            # Find word
            output_word = ""
            for word, index in word_index.items():
                if index == predicted_word_index:
                    output_word = word
                    break
            
            if output_word:
                seed_text += " " + output_word
        
        return json.dumps({'generated_text': seed_text})
    
    except Exception as e:
        return json.dumps({'error': str(e)})
'''
    
    # Save scoring script
    script_dir = 'deployment'
    os.makedirs(script_dir, exist_ok=True)
    
    with open(os.path.join(script_dir, 'score.py'), 'w') as f:
        f.write(scoring_script)
    
    print("Scoring script created")

In [ ]:
# Create environment
if DEPLOY_MODEL:
    from azureml.core.conda_dependencies import CondaDependencies
    
    env = Environment(name='lyrics-model-env')
    
    conda_dep = CondaDependencies()
    conda_dep.add_pip_package('tensorflow>=2.10.0,<2.13.0')
    conda_dep.add_pip_package('numpy')
    conda_dep.add_pip_package('azureml-defaults')
    
    env.python.conda_dependencies = conda_dep
    
    print("Environment created")

In [ ]:
# Configure deployment
if DEPLOY_MODEL:
    inference_config = InferenceConfig(
        entry_script='score.py',
        source_directory=script_dir,
        environment=env
    )
    
    # ACI deployment configuration
    deployment_config = AciWebservice.deploy_configuration(
        cpu_cores=1,
        memory_gb=2,
        tags={'model': 'lyrics-generation', 'type': 'demo'},
        description='Lyrics text generation endpoint'
    )
    
    print("Deployment configuration ready")

In [ ]:
# Deploy the model using global configuration
if DEPLOY_MODEL:
    print(f"Deploying model to {DEPLOYMENT_SERVICE_NAME}...")
    print("This may take several minutes...")
    
    service = Model.deploy(
        workspace=ws,
        name=DEPLOYMENT_SERVICE_NAME,
        models=[registered_model],
        inference_config=inference_config,
        deployment_config=deployment_config,
        overwrite=True
    )
    
    service.wait_for_deployment(show_output=True)
    
    print(f"\nDeployment successful!")
    print(f"Scoring URI: {service.scoring_uri}")
    print(f"Swagger URI: {service.swagger_uri}")

In [ ]:
# Test the deployed endpoint
if DEPLOY_MODEL:
    import requests
    
    test_data = {
        'seed_text': 'love',
        'next_words': 15
    }
    
    headers = {'Content-Type': 'application/json'}
    
    response = requests.post(
        service.scoring_uri,
        data=json.dumps(test_data),
        headers=headers
    )
    
    print("Test request:")
    print(json.dumps(test_data, indent=2))
    print("\nResponse:")
    print(response.json())

## Summary

This notebook has:
1. ✅ Connected to Azure ML Workspace
2. ✅ Loaded lyrics from Azure Blob Storage
3. ✅ Preprocessed the text data
4. ✅ Trained an LSTM text generation model
5. ✅ Exported and registered the model in Azure ML
6. ✅ (Optional) Deployed the model to Azure ML endpoint

### Next Steps:
- Fine-tune model hyperparameters
- Increase training data
- Experiment with different architectures (GPT, BERT, etc.)
- Deploy to production endpoint with authentication